## EXPLORATORY DATA ANALYSIS OF EVENTSHIELD365

The main task of this notebook is to perform EDA on the raw data
collected from both project data sources — event data from
**Ticketmaster** and historical weather data from the **Open-Meteo**
open-source API.

This analysis is intended to build a hands-on understanding of the raw
data before transformation — specifically checking data volume per
city, missing or unreliable fields, and any structural inconsistencies
between sources. Documenting these limitations and boundaries here
ensures they are accounted for during Transform and Load, rather than
discovered later inside the pipeline.

**PHASE 1 :   LOADING THE DATA**

In this phase we will load the data first and then after loading before analyzing things for which our approach is to use the python's libraries like **Pandas** , **Numpy** & **JSON**  , using them we will load the data in the note book 

In [19]:
import pandas as pd  
import json 
import numpy as np  
from pathlib import Path


event_files = list(Path("../data/raw/raw_events").glob("*.json"))
print(f"Found {len(event_files)} event files")

all_events_data = {}
for file_path in event_files:
    with open(file_path, "r") as file:
        all_events_data[file_path.name] = json.load(file)

print(all_events_data.keys())



weather_files = list(Path("../data/raw/raw_weather").glob("*.json"))
print(f"Found {len(weather_files)} weather files")

all_weather_data = {}
for file_path in weather_files:
    with open(file_path, "r") as file:
        all_weather_data[file_path.name] = json.load(file)

print(all_weather_data.keys())

Found 5 event files
dict_keys(['events_Chicago_2026-09-21.json', 'events_LasVegas_2026-09-21.json', 'events_LosAngeles_2026-09-21.json', 'events_Miami_2026-09-21.json', 'events_NewYork_2026-09-21.json'])
Found 5 weather files
dict_keys(['weather_Chicago_2016_2025.json', 'weather_LasVegas_2016_2025.json', 'weather_LosAngeles_2016_2025.json', 'weather_Miami_2016_2025.json', 'weather_NewYork_2016_2025.json'])


The loading part of the data was successful as we can observe in the output cell. Now the next task is to check the events count per city.

**PHASE 2: CHECKING THE INSIGHTS**

1) THIS ONE IS FOR THE EVENTS , WE ARE CHECKING THE EVENT COUNT PETR CITY IN ORDER TO KNOW IF THERE ARE ANY CITIES WITH ABNORMAL VALUES , EITHER TOO HIGH OR NO VALUES FOR A LONGER TIME PERIOD

In [ ]:
sample_filename = list(all_events_data.keys())[0]
sample_events = all_events_data[sample_filename]

events_list = sample_events["events"]   
print(len(events_list))

200


From This we can observe that each city has 200 events , which means there are no outliers in this field of the event data , now we can work with this , but before that we will have a look at the weather data as well.

In [5]:
# Look at one weather file's keys to understand its shape
sample_weather = list(all_weather_data.values())[0]
print(sample_weather.keys())

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])


Now we will check if windgusts_10m_max actually returned real data

In [6]:
sample_daily = sample_weather["daily"]
print(sample_daily.keys())
print(sample_daily["windgusts_10m_max"][:10])  # first 10 values

dict_keys(['time', 'precipitation_sum', 'apparent_temperature_max', 'weathercode', 'windgusts_10m_max'])
[49.3, 51.5, 43.6, 48.6, 38.5, 33.8, 31.3, 23.8, 51.5, 59.0]


AND YES IT DOES RETURN THE REAL VALUES HENCE THE DATA IS PERFECT TILL NOW 

**PHASE 3 : DATA DESCRIPTION**

Now in this Phase we will be checking the basic informations about the data , before this phase we were just taking a look at the loaded data that if it is correctly formated , now when it is confirmed , we can now check the shape , size and other information about the data .

In [25]:
# THIS IS FOR THE WEATHER DATA 

import pandas as pd

filenames = list(all_weather_data.keys())
first_file = filenames[0]

weather_json = all_weather_data[first_file]
daily_data = weather_json["daily"]

weather_df = pd.DataFrame(daily_data)

print("THE NUMBER OF ROWS AND COLUMNS ARE : " ,weather_df.shape)
weather_df.head()


THE NUMBER OF ROWS AND COLUMNS ARE :  (3653, 5)


,time,precipitation_sum,apparent_temperature_max,weathercode,windgusts_10m_max
0,2016-01-01,0.0,-7.3,3,49.3
1,2016-01-02,0.0,-5.4,1,51.5
2,2016-01-03,0.0,-7.0,3,43.6
3,2016-01-04,0.0,-6.8,71,48.6
4,2016-01-05,0.0,-4.8,3,38.5


In [ ]:
# THIS IS FOR THE EVENT DATA 

filenames = list(all_events_data.keys())
first_file = filenames[0]

events_json = all_events_data[first_file]
events_list = events_json["events"]   

names = []
dates = []
segments = []
prices = []

for event in events_list:
    names.append(event.get("name"))
    dates.append(event["dates"]["start"]["localDate"])
    
    if event.get("classifications"):
        segments.append(event["classifications"][0]["segment"]["name"])
    else:
        segments.append(None)
    
    if event.get("priceRanges"):
        prices.append(event["priceRanges"][0]["min"])
    else:
        prices.append(None)

events_df = pd.DataFrame({
    "name": names,
    "date": dates,
    "segment": segments,
    "min_price": prices
})

print(events_df.shape)
events_df.head()

(200, 4)


,name,date,segment,min_price
0,Chicago Bulls vs. Boston Celtics,2026-12-30,Sports,None
1,Chicago Bulls vs. Boston Celtics,2027-03-09,Sports,None
2,Chicago Blackhawks vs. Boston Bruins,2027-01-01,Sports,None
3,Chicago Bulls vs. New York Knicks,2026-10-28,Sports,None
4,Chicago Bulls vs. Los Angeles Lakers,2027-01-02,Sports,None


In [21]:
segments_found = []
for event in events_list:
    segment = event.get("classifications", [{}])[0].get("segment", {}).get("name")
    segments_found.append(segment)

from collections import Counter
print(Counter(segments_found))

Counter({'Sports': 111, 'Music': 52, 'Arts & Theatre': 34, 'Miscellaneous': 2, 'Film': 1})


In [24]:
from collections import Counter

for filename, content in all_events_data.items():
    events_list = content["events"]
    
    segments_found = []
    for event in events_list:
        segment = event.get("classifications", [{}])[0].get("segment", {}).get("name")
        segments_found.append(segment)
    
    print(filename)
    print(Counter(segments_found))
    print("---")

events_Chicago_2026-09-21.json
Counter({'Sports': 111, 'Music': 52, 'Arts & Theatre': 34, 'Miscellaneous': 2, 'Film': 1})
---
events_LasVegas_2026-09-21.json
Counter({'Music': 100, 'Sports': 74, 'Arts & Theatre': 26})
---
events_LosAngeles_2026-09-21.json
Counter({'Sports': 162, 'Arts & Theatre': 35, 'Music': 3})
---
events_Miami_2026-09-21.json
Counter({'Sports': 155, 'Music': 38, 'Miscellaneous': 4, 'Arts & Theatre': 3})
---
events_NewYork_2026-09-21.json
Counter({'Sports': 124, 'Arts & Theatre': 75, 'Music': 1})
---


From the above we loop through all 5 loaded files in all_events_data  and for each one:

Pulls out its events list


Counts the segments, same logic as your single-city check


Prints the filename and its segment breakdown, then a separator line before moving to the next city

**PHASE 4 : MISSING DATA**

In this phase we will be checking data from both the sources that if they have any missing value , any null values or any outliers

In [27]:
sample_filename = list(all_events_data.keys())[0]
events_list = all_events_data[sample_filename]["events"]

missing_date = sum(1 for e in events_list if not e.get("dates", {}).get("start", {}).get("localDate"))
missing_venue = sum(1 for e in events_list if not e.get("_embedded", {}).get("venues"))

print(f"Missing date: {missing_date}")
print(f"Missing venue: {missing_venue}")

Missing date: 0
Missing venue: 0


From the above analysis we can clearly notice that there are 0 missing date and venue data in the event data by the ticketmaster

Now we will have a look at the weather data , if it is having any potential barriers for us .


In [28]:
sample_filename = list(all_weather_data.keys())[0]
sample_weather = all_weather_data[sample_filename]

daily = sample_weather["daily"]
print(daily.keys())

wind_gusts = daily["windgusts_10m_max"]
print(wind_gusts[:10])

none_count = sum(1 for v in wind_gusts if v is None)
print(f"Missing values: {none_count} out of {len(wind_gusts)}")

dict_keys(['time', 'precipitation_sum', 'apparent_temperature_max', 'weathercode', 'windgusts_10m_max'])
[49.3, 51.5, 43.6, 48.6, 38.5, 33.8, 31.3, 23.8, 51.5, 59.0]
Missing values: 0 out of 3653


Now ,again from the code snippet's output its clear that the data is clean  i.e., it has 0 missing values in both the weather data as well as the event data
